# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and analyzing the FAIR^2 clinical colorectal cancer dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

The notebook will guide you through:
- Loading the dataset and Croissant schema
- Viewing dataset metadata and schema structure
- Exploring available record sets, fields, columns (`@id` references)
- Extracting tabular data into pandas DataFrames
- Performing exploratory data analysis and visualization

### Dataset Source
- **Croissant schema URL**: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

All schema elements (record sets, fields, etc.) are referenced **by their `@id`** for best practices with the Croissant specification.

In [ ]:
# Install mlcroissant (if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load the dataset and its metadata from the Croissant schema URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant package
dataset = mlc.Dataset(croissant_url)

# Access dataset-level metadata as an object (not a dictionary)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date published: {metadata.datePublished}")
# Print key dataset fields
print(f"Available record sets: {getattr(metadata, 'recordSet', [])}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s in the schema.

In [ ]:
from mlcroissant.types import RecordSet

# List all record sets in the dataset (by @id and name)
record_sets = [rs for rs in dataset.record_sets]
if not record_sets:
    print("No record sets defined in metadata. Attempting to infer from data distributions...")
    # Try to access distributions and infer record sets/files
    print("Distributions (data files):")
    for dist in getattr(metadata, 'distribution', []):
        print(f"- @id: {getattr(dist, '@id', dist)}")
else:
    print("Record sets available:")
    for rs in record_sets:
        print(f"  • @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")

# For demonstration, check possible record sets from datafiles
from pprint import pprint
print("\nAvailable fields/columns in each record set (detailed view):")

# If there are no record sets, we may explore the files directly as record sets. For this dataset, examine distribution IDs.
distribution_ids = []
for dist in getattr(metadata, 'distribution', []):
    did = getattr(dist, '@id', dist)
    distribution_ids.append(did)
    print(f"- distribution @id: {did}")

## 3. Data Extraction
Load data from available record sets (using `@id`) into pandas DataFrames for analysis.
If record sets are not specified in the Croissant schema, we use the `distribution` objects (which represent data files/tables) as record set references by their `@id`s.

In [ ]:
dataframes = {}

# Use distribution @ids as record set surrogates
record_set_ids = distribution_ids
print("\nLoading available record sets by their @id:")
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"- {record_set_id}: [No records found]")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"- {record_set_id}: loaded {len(df)} records, columns: {list(df.columns)}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

if dataframes:
    # Pick the largest loaded DataFrame for demonstration
    largest_rs = max(dataframes.items(), key=lambda x: x[1].shape[0])[0]
    print(f"\nColumns for record set: {largest_rs}:\n{dataframes[largest_rs].columns.to_list()}")
    print("\nPreview (first 5 rows):")
    display(dataframes[largest_rs].head())
else:
    print("No dataframes loaded. Please check schema or data availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common analysis steps: filter records, normalize numeric fields, categorize or group data. All references to columns must use their `@id`, as provided by the Croissant loader.

In [ ]:
# Pick first loaded record set and examine numeric fields by their @id
if dataframes:
    record_set_id = largest_rs
    df = dataframes[record_set_id].copy()

    # Show list of column @ids
    print(f"Columns (@id) in record set '{record_set_id}':\n{list(df.columns)}\n")

    # Try to pick a likely numeric field (age, interval, etc.)
    from numpy import number
    numeric_candidate = None
    for col in df.columns:
        # Check type by inspecting a sample
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break
    if not numeric_candidate:
        # Try to convert plausible columns to numeric.
        for col in df.columns:
            try:
                conv = pd.to_numeric(df[col], errors='coerce')
                if conv.notna().sum() > 0:
                    numeric_candidate = col
                    df[col] = conv
                    break
            except Exception:
                continue

    if numeric_candidate:
        print(f"Selected numeric field for EDA: '{numeric_candidate}' (referenced by @id).")
        threshold = df[numeric_candidate].quantile(0.75)  # Use 75th percentile as example cut
        filtered_df = df[df[numeric_candidate] > threshold].copy()
        print(f"Filtered records with '{numeric_candidate}' > {threshold} (showing 5):")
        print(filtered_df.head())

        # Normalize numeric field (z-score)
        norm_col = f"{numeric_candidate}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
        print(f"\nNormalized '{numeric_candidate}':\n", filtered_df[[numeric_candidate, norm_col]].head())

        # Attempt grouping by a key categorical field
        # Prefer a field with low cardinality other than numeric_candidate
        group_field = None
        for col in df.columns:
            if col != numeric_candidate and df[col].dtype=='object' and df[col].nunique() < min(8, len(df)//5):
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().to_frame(name=f"mean_{numeric_candidate}")
            print(grouped_df.head())
        else:
            print("No suitable categorical grouping field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and, if possible, its relationship with a grouping variable. All axes and labels use the record set and field `@id` as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and numeric_candidate:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_candidate], bins=10, kde=True)
    plt.title(f"Distribution of '{numeric_candidate}' in filtered records (record set: {record_set_id})")
    plt.xlabel(numeric_candidate)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_candidate])
        plt.title(f"'{numeric_candidate}' by '{group_field}' (record set: {record_set_id})")
        plt.xlabel(group_field)
        plt.ylabel(numeric_candidate)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, you have:
- Loaded the FAIR^2 clinical colorectal cancer dataset from its Croissant schema
- Explored the dataset's available data distributions via their `@id` references
- Loaded tabular data using `mlcroissant`, referencing all record sets and columns by `@id`
- Carried out example data filtering, normalization, grouping, and visualization steps

**Next steps:**
- Examine the dataset schema in detail for specialized analyses
- Map field `@id`s to clinical concepts as needed using metadata
- Apply statistical or machine learning methods on the processed DataFrames

For more, see the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) and the dataset's published metadata.